In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from src.calibration import IsotonicCalibrator, PlattScaling
from src.data import DATASETS, RANDOM_STATE, load_splits
from src.metrics import evaluate


sns.set_theme(style="whitegrid", context="paper")
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = ROOT / "report" / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "LR": lambda: LogisticRegression(max_iter=1000),
    "RF": lambda: RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE),
}

In [2]:
data = {}
for name in DATASETS:
    X_train, X_cal, X_test, y_train, y_cal, y_test = load_splits(name)
    data[name] = {"train": (X_train, y_train), "cal": (X_cal, y_cal), "test": (X_test, y_test)}

In [3]:
models = {}
for name in DATASETS:
    X, y = data[name]["train"]
    for model_name, make in MODELS.items():
        model = make().fit(X, y)
        models[name, model_name] = model
        iterations = f"  ({model.n_iter_[0]} iterations)" if model_name == "LR" else ""
        print(f"{name:10s} {model_name}: {model}{iterations}")

higgs      LR: LogisticRegression(max_iter=1000)  (17 iterations)
higgs      RF: RandomForestClassifier(n_jobs=-1, random_state=42)
covertype  LR: LogisticRegression(max_iter=1000)  (93 iterations)
covertype  RF: RandomForestClassifier(n_jobs=-1, random_state=42)
adult      LR: LogisticRegression(max_iter=1000)  (82 iterations)
adult      RF: RandomForestClassifier(n_jobs=-1, random_state=42)


In [4]:
def model_outputs(model, model_name, X):
    proba = model.predict_proba(X)[:, 1]                         
    score = model.decision_function(X) if model_name == "LR" else proba
    return score, proba


outputs = {}
for (name, model_name), model in models.items():
    for split in ("cal", "test"):
        outputs[name, model_name, split] = model_outputs(model, model_name, data[name][split][0])

np.savez_compressed(
    RESULTS_DIR / "uncalibrated_outputs.npz",
    **{f"{n}_{m}_{s}_{kind}": arr
       for (n, m, s), pair in outputs.items() for kind, arr in zip(("score", "proba"), pair)})